# Yambda AutoFE — запуск на Kaggle

Настройки ноутбука: **Internet = On**, Accelerator = None (CPU достаточно). Ноутбук клонирует репозиторий, скачивает Yambda-50M с HuggingFace, прогоняет тесты на утечки и все эксперименты. Результаты — в `/kaggle/working/recsys/results` (их нужно скачать: Output → Download, либо архив `results.zip`).

In [ ]:
!git clone -q https://github.com/georgiken/yambda-autofe.git recsys
%cd recsys
!pip install -q -r requirements.txt 2>&1 | tail -2
!nproc; free -g

In [ ]:
# 1) тесты на утечки (синтетика, ~15 с)
!python -m pytest -q tests

In [ ]:
# 2) основной эксперимент: все 10k пользователей Yambda-50M (~1 ч на CPU)
!python scripts/run_experiments.py --out results/u10000 2>&1 | grep -v Warning

In [ ]:
# 3) масштабируемость: 1k и 3k случайных пользователей с полной историей
!python scripts/run_experiments.py --n-users 1000 --out results/u1000 2>&1 | grep -v Warning
!python scripts/run_experiments.py --n-users 3000 --out results/u3000 2>&1 | grep -v Warning

In [ ]:
# 4) таблицы и графики + архив для скачивания
!python scripts/make_report.py results
!rm -f results/events_*.parquet && cd results && zip -qr ../results.zip . && ls -la ..